# Preparación, variables y contrato de evaluación

Estos notebooks documentan experimentos ejecutados y leen sus artefactos. El entrenamiento completo se reproduce con `scripts/run_experiments.py`; las celdas de revisión no ajustan modelos ni cambian la selección. El informe final y la presentación corresponden a una etapa posterior.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
OUT = ROOT / 'outputs'
pd.set_option('display.max_columns', 15)
pd.set_option('display.max_rows', 30)
protocol = json.loads((ROOT/'experiments'/'protocol.json').read_text(encoding='utf-8'))

## Protocolo fijado antes de entrenar

**Alcance de la evaluación:** clasificación retrospectiva de `Charged Off` frente a `Fully Paid` entre préstamos aceptados con resultado registrado. El split es por fecha de emisión, con IDs externos a la muestra inicial en 2015 y 2016. El snapshot no reconstruye todas las fechas históricas de disponibilidad de las etiquetas. Deben leerse la censura y los plazos junto a las métricas.

In [2]:
display(pd.DataFrame(protocol['cv']))
display(pd.DataFrame([{k: protocol[k] for k in ['candidate_training_end','selection_end','final_training_end','calibration_start','calibration_end','policy_start','policy_end']}]).T)

,train_end,validation_start,validation_end
0,2011-12-31,2012-01-01,2012-12-31
1,2012-12-31,2013-01-01,2013-12-31


,0
candidate_training_end,2013-12-31
selection_end,2014-06-30
final_training_end,2014-06-30
calibration_start,2014-07-01
calibration_end,2014-09-30
policy_start,2014-10-01
policy_end,2014-12-31


## Trazabilidad de los datos

El archivo original no se añade a Git por su tamaño. El manifiesto contiene su SHA-256, las huellas de los IDs y el solapamiento con la muestra previa. El pipeline comprueba que los 115.000 IDs de esa muestra estén en el archivo original.

In [3]:
manifest = json.loads((OUT/'data_manifest.json').read_text(encoding='utf-8'))
display(pd.DataFrame(manifest['partitions']).T)
print('Filas de la fuente:', manifest['source_rows'])
print('SHA-256:', manifest['source_sha256'])
print('IDs adicionales excluidos por inspección de esquema:',manifest['schema_preview_excluded_ids'])

,rows,ids_sha256,file_sha256,original_sample_overlap,schema_preview_overlap
development,451059,4b3429e3cebcbc79a8fda730d887f9f806577f09079bec...,6b065a38a544d3601bd537393c78f5cfbe19bd84efbd70...,38592,NaN
holdout_2015,343507,707ff67445186f2a0876fb3fee431be72334e737932027...,24c604b89693d9bc9d8e6fc5647d179297ed01b583b565...,0,0
holdout_2016,268096,497a8703fc67ebc5f6f24e9d69dbb8f17db96c1681785c...,682113d4acfb7bf834875c9e2a4c37000361b66e0d6a5f...,0,0


Filas de la fuente: 2260701
SHA-256: 55c16f75120f897683f02e7aabcf080d0e4a20c4832feb1d592cfa941bd62a2d
IDs adicionales excluidos por inspección de esquema: ['68341763', '68355089', '68407277', '68476715']


## Auditoría de variables

Ingreso no positivo, FICO imposible y valores negativos incompatibles con el dominio se convierten a ausentes mediante reglas fijas. Se conservan extremos plausibles y se añaden razones, logaritmos e historia crediticia. Imputación, codificación y escalado se ajustan únicamente en entrenamiento. Los campos totalmente ausentes o constantes en entrenamiento se identifican aquí; disponer de más columnas en años posteriores no permite aprenderlas retrospectivamente.

In [4]:
audit = pd.read_csv(OUT/'analysis'/'feature_audit.csv')
print('Variables del conjunto ampliado:', len(audit))
print('No informativas en entrenamiento:', int(audit.uninformative_on_train.sum()))
display(audit.sort_values('missing_fraction_train', ascending=False))

Variables del conjunto ampliado: 96
No informativas en entrenamiento: 17


,feature,missing_fraction_train,unique_values_train,origin,uninformative_on_train
24,open_il_24m,1.0,0,origination_raw,True
25,mths_since_rcnt_il,1.0,0,origination_raw,True
26,total_bal_il,1.0,0,origination_raw,True
31,all_util,1.0,0,origination_raw,True
27,il_util,1.0,0,origination_raw,True
...,...,...,...,...,...
89,loan_to_income,0.0,59894,engineered_origination_only,False
91,open_to_total_accounts,0.0,1047,engineered_origination_only,False
90,revolving_debt_to_income,0.0,260155,engineered_origination_only,False
93,loan_to_monthly_income,0.0,59894,engineered_origination_only,False


## Exclusiones y disponibilidad

Se excluyen estados finales, pagos, recuperaciones, cobranza, hardship, settlement, últimas consultas/FICO posteriores al otorgamiento e identificadores. También grade, sub_grade, tasa, cuota y financiación asignados por Lending Club. La verificación se considera disponible al completar la evaluación de la solicitud. `issue_d` solo calcula la antigüedad crediticia; no entra directamente en el modelo. Los campos del bureau describen información previa y no la historia posterior de este préstamo.

## Cobertura de resultados terminales

La siguiente tabla se liberó después de congelar el modelo. La fracción no terminal excluida cuantifica por qué los resultados de cohortes recientes o a 60 meses requieren cautela.

In [5]:
display(pd.read_csv(OUT/'final'/'maturity_coverage.csv'))

,year,term,external_population_n,resolved_n,excluded_nonterminal_n,terminal_fraction
0,2015,36 months,259003,258856,147,0.999432
1,2015,60 months,130054,84651,45403,0.650891
2,2016,36 months,303624,212482,91142,0.699820
3,2016,60 months,105784,55614,50170,0.525732


## Fuentes de investigación

La bitácora con referencias a artículos y decisiones está en `experiments/research_notes.md`. Las hipótesis de esos artículos se contrastan aquí con datos del proyecto; no se copian sus métricas como objetivos.